# Module 07 — Strings and formatting

Most of this module is notation you will pick up in an afternoon. One section is
not: `==` and `is` mean the opposite of what they mean in Java, and Python is
generous enough with the wrong one to let it pass its first test.

## 1. A string is an immutable sequence

Immutable, as in Java. A sequence, so everything module 05 said about indexing and
slicing applies unchanged.

In [ ]:
tag = "TH-04"

print(tag[0], tag[-1], tag[:2], tag[::-1])
print(len(tag), "H-0" in tag, "-" * 5)

try:
    tag[0] = "X"
except TypeError as err:
    print("TypeError:", err)

One difference from module 05 worth noticing now, because section 2 turns on it:
`v[:]` on a list builds a copy, and `s[:]` on a string does not have to — there is
nothing to protect, so Python hands the same object back.

In [ ]:
values = [1, 2]
text = "TH-04"

print(values[:] is values)  # a copy: a list can change, so it must be one
print(text[:] is text)  # the same object: a string cannot change

## 2. `==` and `is` — the inversion

This is the section. In Java, `==` on two strings compares references and
`.equals()` compares content, which is why `str1 == str2` is a bug you learned to
spot on sight. Python has the same two questions and the opposite spelling:

| | compares content | compares identity |
| --- | --- | --- |
| Java | `a.equals(b)` | `a == b` |
| Python | `a == b` | `a is b` |

So `==` is the one you want, always. What makes this a trap rather than a
translation table is that `is` often looks right.

In [ ]:
a = "TH-04"
b = "TH-04"

print("two literals:      ", a is b, a == b)

parts = ["TH", "-04"]
built = "".join(parts)  # the same characters, assembled while the program runs

print("built at runtime:  ", built is a, built == a)

Python keeps one copy of each string literal in a program — the compiler sees them
in the source and reuses the object. Two literals that read the same therefore *are*
the same object, and `is` says `True`.

A string assembled while the program runs is a new object. `is` then says `False`,
while `==` still says `True`, because the content is what `==` looks at.

Which means: `is` on strings agrees with `==` exactly as long as your test data is
written in the source, and starts disagreeing the moment the data comes from a file,
a socket or a user. Predict both lines below.

In [ ]:
name = "TH-04"

from_source = "TH-04"
from_input = "".join(["TH", "-04"])  # what reading a line would give you

assert (name is from_source) == ...
assert (name is from_input) == ...

The rule that follows is short: **compare strings with `==`.** `is` has exactly one
job — asking whether two names refer to one object — and `x is None` is where you
will actually use it.

Section 4 has one more reason not to trust `is` here.

## 3. f-strings

A leading `f` turns the braces in a string into expressions. This replaces
`String.format`, concatenation with `+`, and the older Python forms in section 8.

In [ ]:
tag = "TH-04"
value = 91.037
limit = 85

print(f"{tag} read {value} (limit {limit})")
print(f"{tag.lower()} is {'above' if value > limit else 'within'} the limit")

After a colon comes the **format spec**: what the value should look like.

In [ ]:
value = 91.037

print(f"[{value:.2f}]")  # two decimals
print(f"[{value:8.2f}]")  # ... in a field eight wide, right-aligned
print(f"[{'TH-04':>10}]")  # right
print(f"[{'TH-04':<10}]")  # left -- the default for text
print(f"[{'TH-04':^10}]")  # centred
print(f"[{'TH-04':*^11}]")  # centred, padded with something other than a space
print(f"[{1234567:,}]")  # thousands separator
print(f"[{0.256:.1%}]")  # as a percentage
print(f"[{5:+d}]")  # always show the sign

Numbers are right-aligned by default and text is left-aligned, which is what you
want for a table and worth knowing before you specify it by hand.

An `=` after the expression prints the expression *and* its value. It is the
shortest debugging tool in the language, and it is not a string you have to keep in
sync with the code — the text comes from the source.

In [ ]:
value = 91.037
readings = [21.7, 23.1]

print(f"{value=}")
print(f"{len(readings)=}")
print(f"{value > 85=}")

## 4. Methods hand back a result

A string cannot change, so nothing here modifies anything: every method returns a
value, and ignoring it is the mistake — the same shape as `x.sort()` in module 05,
arrived at from the other side.

In [ ]:
line = "  TH-04 ; 91.0  "

print(repr(line.strip()))
print(repr(line))  # unchanged, as it has to be
print("TH-04".replace("-", "_"), "th-04".upper(), "TH-04".startswith("TH"))
print("TH-04".removeprefix("TH-"))  # 3.9 and later; cleaner than a slice with a magic 3

Here is the last nail in section 2's coffin. When a method has nothing to do,
CPython may hand you back the object you passed in rather than build an identical
one — an optimisation you are not supposed to notice.

In [ ]:
tag = "TH-04"
padded = "  x  "

print(tag.strip() is tag)  # nothing to strip -- the same object comes back
print(padded.strip() is padded)  # something to strip -- a new one
print(tag.upper() is tag)  # already upper case, and still a new object

So `is` on strings answers a question about how CPython happened to allocate memory.
That is not the question you meant to ask.

Python says so itself: write the literal directly into the comparison — `text is
"TH-04"` — and the interpreter emits `SyntaxWarning: "is" with 'str' literal. Did
you mean "=="?`. The warning fires only when the literal is right there in the line,
which is why the examples here go through a name: the trap survives exactly where
the warning cannot see it.

## 5. Taking a line apart

`.split()` with no argument splits on **runs** of whitespace and throws away the
empty pieces. `.split(sep)` splits on exactly that separator and keeps them. Java's
`split(" ")` behaves like the second one, so this is a real difference and not a
detail.

In [ ]:
line = "  TH-04   91.0   C  "

print(line.split())  # runs of whitespace, no empties
print(line.split(" "))  # every single space is a separator

record = "TH-04;91.0;C"
print(record.split(";"))
print(record.split(";", 1))  # at most one split -- the rest stays whole

tag, value, unit = record.split(";")  # module 05's unpacking, on the result
print(tag, value, unit)

`.join` is the inverse, and it is a method **on the separator**, which reads
backwards until you have written it twice. It needs strings — it will not convert
for you.

In [ ]:
fields = ["TH-04", "91.0", "C"]

print(";".join(fields))
print(", ".join(fields))
print("".join(fields))

try:
    ";".join(["TH-04", 91.0])
except TypeError as err:
    print("TypeError:", err)

## 6. Building a string in a loop

`+=` on a string cannot append: the string is immutable, so each pass builds a whole
new one and throws the old away. That is the loop Java tells you to write a
`StringBuilder` for, and `"".join()` is the answer here.

In [ ]:
fields = ["TH-04", "91.0", "C"]

# The shape to avoid: a new string on every pass, plus a separator to trim afterwards.
line = ""
for field in fields:
    line += field + ";"
print(repr(line.removesuffix(";")))

# What to write instead.
print(repr(";".join(fields)))

# join takes any iterable, so a comprehension goes straight in.
readings = [21.7, 91.0]
print(";".join(f"{r:.1f}" for r in readings))

For three fields the difference is style. For a loop over a large file it is the
difference between linear and quadratic work, which is exactly why Java gives you a
separate class for it.

The `f"{r:.1f}"` inside `join` is also the answer to "join needs strings": convert
where you build the pieces, not by patching afterwards.

## 7. What `len` counts

A Python `str` is a sequence of **code points**. `len` counts those. It does not
count bytes, and it does not count what Java counts.

In [ ]:
for text in ("TH-04", "Übergabe", "👍"):
    print(f"{text!r:12} len={len(text)}  bytes={len(text.encode('utf-8'))}")

Two separate things to take from that:

- **Bytes are a different question.** `.encode("utf-8")` turns a `str` into `bytes`,
  and `bytes.decode("utf-8")` turns it back. `"ü"` is one code point and two bytes.
  Files (module 08) and HTTP (module 16) hand you bytes; everything in between should
  be `str`.
- **Java would say 2 for the emoji.** A Java `String` is UTF-16 code units, and
  anything outside the basic plane takes two of them. Python counts the character.

`!r` in the f-string above asks for `repr` instead of `str` — quotes and escapes
visible. It is what you want when the question is "what exactly is in there".

In [ ]:
text = "Übergabe"
raw = text.encode("utf-8")

print(raw)
print(type(raw), len(raw))
print(raw.decode("utf-8") == text)

## 8. Two forms you will read and not write

Older code formats with `%` or with `.format()`. Both still work, and you will meet
both — `%` in logging calls especially, where it is still the recommended form
because the formatting only happens if the message is actually emitted.

`ruff` reports both lines below (`UP031`, `UP032`) and offers to rewrite them as
f-strings. That report is the evidence for what this section claims: the tooling
treats these as forms to be migrated, so read them and write the third one. The
`# noqa` comments keep the examples from being rewritten under you.

In [ ]:
tag, value = "TH-04", 91.037

print("%s read %.2f" % (tag, value))  # noqa: UP031 -- shown, not recommended
print("{} read {:.2f}".format(tag, value))  # noqa: UP032 -- likewise
print(f"{tag} read {value:.2f}")  # what to write

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run.

Module 08 takes strings to where they come from: a file on disk, with an encoding
that has to be named.